In [1]:
import os
import ast
import pandas as pd
from Bio import Phylo
import rpy2.robjects as robjects
from tqdm import tqdm
import multiprocessing
import time
import hashlib
import warnings

warnings.filterwarnings('ignore')

def get_all_clades(clade, clade_list=None):
    if clade_list is None:
        clade_list = []
    if not clade.is_terminal():
        clade_list.append(clade)
        for child in clade.clades:
            get_all_clades(child, clade_list)
    return clade_list

def get_clade_signature(clade):
    leaves = sorted([leaf.name for leaf in clade.get_terminals()])
    leaf_hash = hashlib.md5(','.join(leaves).encode()).hexdigest()
    signature = f"{leaf_hash}"
    return signature, leaves

def add_bootstrap_to_tree(tree, original_signatures_counts, n_bootstrap):
    def _recursive_add(clade):
        if not clade.is_terminal():
            sig, _ = get_clade_signature(clade)
            support = original_signatures_counts[sig]/n_bootstrap*100
            clade.name = f"{support:.1f}"
            for child in clade.clades:
                _recursive_add(child)
    _recursive_add(tree.root)

def bootstrap_analysis(que, folder, boot_folder, table_name, tree_name, method, index, num, n_bootstrap=1000):
    #num_list = []
    count = index
    robjects.r(f'''
    library("ape")
    setwd("{folder}")
    data <- read.csv(file = "{table_name}", row.names = 1, header=T)
    n_features <- ncol(data)
    ''')
    while count < n_bootstrap:
        robjects.r(f'''
        feature_idx <- sample(1:n_features, size = n_features, replace = TRUE)
        data_boot <- data[, feature_idx]
        da <- dist(data_boot)
        hc <- do.call("hclust", list(da, method="{method}"))
        tr <- as.phylo(hc)
        setwd("{boot_folder}")
        write.tree(tr, file = "{tree_name}-{count}.txt",)
        ''')
        que.put(count)
        count += num

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
def multiple_process(genus_name, label, table_type, normalized = True, method = 'average', par = 16, n_bootstrap = 1000):
    folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/tree_data'
    boot_folder = f'{folder}/bootstrap_samples'
    if not os.path.exists(boot_folder):
        os.makedirs(boot_folder)
    
    os.chdir(folder)
    table_name = f"{table_type}_bitscore-{label}-normalized_by_size.csv" if normalized else f"{table_type}_bitscore-{label}.csv"
    tree_name = f"tr_{method}-{table_type}-{label}-normalized_by_size.txt" if normalized else f"tr_{method}-{table_type}-{label}.txt"
    original_tree = Phylo.read(tree_name, "newick")

    original_clades = get_all_clades(original_tree.root)
    original_signatures = {}  # {signature: leaves}
    bootstrap_counts = {}
    for clade in original_clades:
        sig, leaves = get_clade_signature(clade)
        original_signatures[sig] = leaves
        bootstrap_counts[sig] = 0
        
    manager = multiprocessing.Manager()
    que = manager.Queue()
    pool = multiprocessing.Pool(par)
    
    for i in range(par):
        pool.apply_async(bootstrap_analysis, (que, folder, boot_folder, table_name, tree_name, method, i, par, n_bootstrap))

    pool.close()
    
    count = 0
    num_list = []
    with tqdm(total = n_bootstrap, desc=f'{genus_name}-{table_type}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            time.sleep(0.01)
            if not que.empty():
                value = que.get(True)
                num_list.append(value)
                count += 1
                pbar.update(1)
                if count == n_bootstrap:
                    break
            else:
                continue
    
    pool.join()
    
    for count in num_list:
        boot_tree = Phylo.read(f'{boot_folder}/{tree_name}-{count}.txt', "newick")
        boot_clades = get_all_clades(boot_tree.root)
        boot_signatures = {}
        for clade in boot_clades:
            sig, leaves = get_clade_signature(clade)
            try:
                bootstrap_counts[sig] += 1
            except:
                pass
    
    add_bootstrap_to_tree(original_tree, bootstrap_counts, n_bootstrap = len(num_list))
    os.chdir(folder)
    b_file = open(f'{tree_name}-bootstrap.txt', 'w+')
    Phylo.write(original_tree, b_file, "newick")
    b_file.close()

In [ ]:
labels = ['original', 'pident_90', 'pident_95']
table_types = ['accession', 'MS_replicon', 'typical_chr_by_acc']

for genus_name in keep_genus:
    method = 'average'
    label = labels[1]
    for table_type in table_types:
        normalized = True
        multiple_process(genus_name, label, table_type, normalized, method, par = 32)

Escherichia-accession: 100%|██████████████████████████████████| 1.00k/1.00k [4:09:32<00:00, 15.0s/B]
Escherichia-MS_replicon: 100%|████████████████████████████████| 1.00k/1.00k [4:07:17<00:00, 14.8s/B]
Escherichia-typical_chr_by_acc: 100%|█████████████████████████| 1.00k/1.00k [4:08:19<00:00, 14.9s/B]
Klebsiella-accession: 100%|███████████████████████████████████| 1.00k/1.00k [2:15:14<00:00, 8.11s/B]
Klebsiella-MS_replicon: 100%|█████████████████████████████████| 1.00k/1.00k [2:12:26<00:00, 7.95s/B]
Klebsiella-typical_chr_by_acc: 100%|██████████████████████████| 1.00k/1.00k [2:12:15<00:00, 7.94s/B]
Staphylococcus-accession: 100%|█████████████████████████████████| 1.00k/1.00k [41:40<00:00, 2.50s/B]
Staphylococcus-MS_replicon: 100%|███████████████████████████████| 1.00k/1.00k [42:45<00:00, 2.57s/B]
Staphylococcus-typical_chr_by_acc: 100%|████████████████████████| 1.00k/1.00k [42:19<00:00, 2.54s/B]
Pseudomonas-accession: 100%|████████████████████████████████████| 1.00k/1.00k [37:56<00:00,

In [4]:
for genus_name in keep_genus[14:]:
    method = 'average'
    label = labels[1]
    for table_type in table_types:
        normalized = True
        multiple_process(genus_name, label, table_type, normalized, method, par = 32)

Vibrio-accession: 100%|█████████████████████████████████████████| 1.00k/1.00k [00:40<00:00, 24.8B/s]
Vibrio-MS_replicon: 100%|███████████████████████████████████████| 1.00k/1.00k [00:38<00:00, 25.7B/s]
Vibrio-typical_chr_by_acc: 100%|████████████████████████████████| 1.00k/1.00k [00:40<00:00, 24.7B/s]
Mycobacterium-accession: 100%|██████████████████████████████████| 1.00k/1.00k [00:33<00:00, 30.1B/s]
Mycobacterium-MS_replicon: 100%|████████████████████████████████| 1.00k/1.00k [00:33<00:00, 29.8B/s]
Mycobacterium-typical_chr_by_acc: 100%|█████████████████████████| 1.00k/1.00k [00:33<00:00, 29.8B/s]
Corynebacterium-accession: 100%|████████████████████████████████| 1.00k/1.00k [00:20<00:00, 49.3B/s]
Corynebacterium-MS_replicon: 100%|██████████████████████████████| 1.00k/1.00k [00:20<00:00, 48.5B/s]
Corynebacterium-typical_chr_by_acc: 100%|███████████████████████| 1.00k/1.00k [00:20<00:00, 48.5B/s]
Burkholderia-accession: 100%|███████████████████████████████████| 1.00k/1.00k [00:15<00:00,